In [2]:
import duckdb
import pandas as pd

con = duckdb.connect(r"..\data\BaseDeDatos_Working.db")

In [30]:
df = con.execute("""
SELECT *
FROM incendios
""").df()

In [31]:
df.columns

Index(['id_clave_inc', 'id_cvegeo', 'id_causa', 'id_vegetacion', 'latitud',
       'longitud', 'fecha_inicio', 'tipo_de_incendio', 'anio'],
      dtype='str')

In [32]:
ignicion_vars = [
    "id_causa",
    "poblacion",  # si existe en demografia
    "densidad",   # ejemplo
]

propagacion_vars = [
    "NDVI",
    "NDWI",
    "NDMI",
    "TSOIL6",
    "T10M_RANGE",
    "TS_RANGE",
    "EVLAND",
    "EVPTRNS",
    "CLRSKY_SFC_LW_DWN",
    "ALLSKY_SFC_LW_UP"
]

impacto_vars = [
    "danos",
    "tipo_de_vegetacion",
    "regimen_del_fuego"
]

In [33]:
con.execute("SELECT * FROM climatologia LIMIT 5").df()

,id_registro,id_variable,id_clave_inc,fecha_de_observacion,resultado_numerico
0,491551,SRF_ALB_ADJ,17-15-0771,2017-03-24,0.13
1,491552,SRF_ALB_ADJ,17-15-0771,2017-03-25,0.14
2,491553,SRF_ALB_ADJ,17-15-0771,2017-03-26,0.12
3,491554,SRF_ALB_ADJ,17-15-0771,2017-03-27,0.15
4,491555,SRF_ALB_ADJ,17-15-0771,2017-03-28,0.14


In [34]:
con.execute("SELECT * FROM demografia LIMIT 5").df()

,id_cvegeo,anio,sexo,pob_total,r_00_04,r_05_09,r_10_14,r_15_19,r_20_24,r_25_29,...,r_40_44,r_45_49,r_50_54,r_55_59,r_60_64,r_65_69,r_70_74,r_75_79,r_80_84,r_85_mm
0,15001,2019,hombres,33228,3617,3521,3672,3401,2949,2616,...,1722,1649,1460,1214,964,758,612,446,278,247
1,15002,2015,hombres,77323,7796,7663,7956,7497,7017,6414,...,5533,4504,3533,2717,1886,1250,877,595,359,295
2,15002,2018,hombres,83837,6731,7318,7967,8159,8023,7703,...,5905,5363,4334,3308,2300,1504,1062,691,399,341
3,15002,2021,mujeres,88986,5957,6694,7813,8329,7987,7714,...,6964,6503,5243,3942,2914,1990,1317,855,496,447
4,15003,2019,hombres,24597,2416,2466,2586,2488,2274,2084,...,1347,1211,1050,915,755,569,418,288,208,179


In [35]:
con.execute("SELECT * FROM vegetacion LIMIT 5").df()

,id_vegetacion,regimen_del_fuego,tipo_de_vegetacion
0,18,otros,mezquital espinoso
1,19,otros,matorral crasicaule
2,20,adaptado,matorral espinoso tamaulipeco
3,21,adaptado,pradera de alta montana
4,22,otros,matorral sarco-crasicaule


In [38]:
df.head()

,id_clave_inc,id_cvegeo,id_causa,id_vegetacion,latitud,longitud,fecha_inicio,tipo_de_incendio,anio
0,15-15-0156,15039,32,28,19.309111,-99.788834,2015-03-02,superficial,2015
1,16-15-0018,15014,30,1,19.877222,-99.875557,2016-01-23,superficial,2016
2,16-15-0153,15060,30,27,19.596666,-99.432281,2016-02-11,superficial,2016
3,16-15-0207,15063,39,1,19.002361,-99.367722,2016-02-16,superficial,2016
4,16-15-0869,15038,30,27,19.574194,-99.395164,2016-04-08,superficial,2016


In [44]:
df = con.execute("""
SELECT 
    id_clave_inc,
    id_variable,
    AVG(resultado_numerico) AS valor
FROM climatologia
GROUP BY id_clave_inc, id_variable
""").df()

df_pivot = df.pivot(
    index="id_clave_inc",
    columns="id_variable",
    values="valor"
).reset_index()

print(df_pivot.shape)
df_pivot.head()

(12586, 123)


id_variable,id_clave_inc,AIRMASS,ALLSKY_KT,ALLSKY_NKT,ALLSKY_SFC_LW_DWN,ALLSKY_SFC_LW_UP,ALLSKY_SFC_PAR_DIFF,ALLSKY_SFC_PAR_DIRH,ALLSKY_SFC_PAR_TOT,ALLSKY_SFC_SW_DIFF,...,WS10M_RANGE,WS2M,WS2M_MAX,WS2M_MIN,WS2M_RANGE,WS50M,WS50M_MAX,WS50M_MIN,WS50M_RANGE,Z0M
0,15-15-0001,3.328,0.654,0.809,23.897,31.743,3.458,4.118,7.576,5.758,...,2.882,1.158,2.180,0.338,1.842,2.616,4.897,0.952,3.945,0.424
1,15-15-0002,3.420,0.638,0.796,24.189,31.832,3.784,3.589,7.377,6.397,...,2.736,1.269,2.440,0.418,2.022,2.797,4.303,0.983,3.320,0.203
2,15-15-0003,3.783,0.657,0.838,24.237,32.195,3.557,4.054,7.614,5.969,...,2.100,0.668,1.322,0.104,1.218,1.884,3.334,0.355,2.979,0.985
3,15-15-0004,3.783,0.657,0.838,24.237,32.195,3.557,4.054,7.614,5.969,...,2.442,1.294,2.369,0.521,1.848,2.968,4.445,1.208,3.237,0.180
4,15-15-0005,3.959,0.661,0.854,24.284,32.246,3.656,4.031,7.691,6.157,...,2.390,0.882,1.828,0.182,1.646,2.095,3.810,0.563,3.247,0.420


In [45]:
df_pivot = df_pivot.rename(columns={"id_clave_inc": "id_clave_inc"})

In [62]:
df_incendios = con.execute("SELECT * FROM incendios").df()

In [63]:
df_final = df_incendios.merge(df_pivot, on="id_clave_inc", how="left")

In [64]:
df_final.columns

Index(['id_clave_inc', 'id_cvegeo', 'id_causa', 'id_vegetacion', 'latitud',
       'longitud', 'fecha_inicio', 'tipo_de_incendio', 'anio', 'AIRMASS',
       ...
       'WS10M_RANGE', 'WS2M', 'WS2M_MAX', 'WS2M_MIN', 'WS2M_RANGE', 'WS50M',
       'WS50M_MAX', 'WS50M_MIN', 'WS50M_RANGE', 'Z0M'],
      dtype='str', length=131)

In [65]:
df_final.shape

(12703, 131)

In [66]:
df_final.isna().sum().sort_values(ascending=False).head(20)

CLRSKY_NKT    1974
PBLTOP        1942
TSOIL2        1942
T10M_RANGE    1942
TO3           1942
TQV           1942
TSOIL5        1942
TSOIL6        1942
TSOIL3        1942
TSOIL4        1942
TS_RANGE      1942
V50M          1942
U2M           1942
V2M           1942
U50M          1942
Z0M           1942
TSURF         1942
TS_ADJ        1942
TROPPB        1942
RHOA          1942
dtype: int64

In [67]:
df_final["indice_propagacion"] = (
    df_final["NDVI"].fillna(0) * 0.4 -
    df_final["NDMI"].fillna(0) * 0.3 -
    df_final["NDWI"].fillna(0) * 0.2 +
    df_final["TSOIL6"].fillna(df_final["TSOIL6"].median()) * 0.1
)

## Analisis por tipo de incendio 

In [68]:
df_final.groupby("tipo_de_incendio")["indice_propagacion"].mean().sort_values()

tipo_de_incendio
superficial    1.723080
mixto          1.734318
de copa        1.743945
subterraneo    1.785958
Name: indice_propagacion, dtype: float64

El índice de propagación muestra un incremento progresivo conforme aumenta la complejidad del incendio (superficial < mixto < copa < subterráneo), lo que refleja que los eventos más críticos están vinculados a una menor humedad del combustible y una mayor carga energética. Sin embargo, dado que el rango de variación es estrecho (1.723 a 1.785), se concluye que, aunque el índice es capaz de discriminar tendencias generales de severidad y disponibilidad biomasa, no debe utilizarse como un criterio único para separar los tipos de incendio, sino como un indicador de condiciones ambientales críticas.

## Analisis por causa del incendio

In [69]:
df_final.groupby("id_causa")["indice_propagacion"].mean()

id_causa
1     1.716506
2     1.768435
3     1.848857
4     1.638215
5     1.687878
6     1.632399
7     1.754335
8     1.720058
9     1.761067
10    1.816461
11    1.878980
12    1.885605
13    1.758692
14    1.656039
15    1.692810
16    1.785957
17    1.981733
18    1.667855
19    1.725910
20    1.746047
21    1.767109
22    1.805770
23    1.710946
24    1.780250
25    1.778876
26    1.712046
27    1.736706
28    1.745597
29    1.800454
30    1.721207
31    1.674126
32    1.759896
33    1.670395
34    1.605551
35    1.677095
36    1.872205
37    1.681546
38    1.652822
39    1.723897
40    1.694693
41    1.820825
42    1.590396
43    1.606738
44    1.723072
45    1.675576
46    1.669704
47    1.581854
48    1.857399
49    1.604416
50    1.758730
51    1.794458
52    1.692635
53    1.707907
54    1.730593
55    1.627849
56    1.903749
57    1.721947
58    1.672135
Name: indice_propagacion, dtype: float64

El análisis del índice de propagación por causa revela una variación significativa (entre 1.63 y 1.98), lo que demuestra que el comportamiento del fuego no es uniforme y depende de la interacción entre el origen del incendio y el entorno. Los resultados sugieren que ciertas causas (como las identificadas con índices superiores a 1.85) están asociadas a escenarios climáticos más críticos de baja humedad y alta disponibilidad de combustible seco, mientras que las causas con índices menores ocurren en condiciones menos extremas. En conclusión, el hallazgo principal es que la causa no determina por sí sola la intensidad del fuego, sino que actúa como un detonante que se potencia según el estado de la vegetación y el clima al momento del evento.

## Analisis por año 

In [70]:
df_final.groupby("anio")["indice_propagacion"].mean()

anio
2015    1.710269
2016    1.676701
2017    1.717903
2018    1.707777
2019    1.716475
2020    1.752080
2021    1.741632
2022    1.778391
2023    1.767791
2024    1.699567
2025    1.702912
Name: indice_propagacion, dtype: float64

El análisis histórico revela una tendencia incremental en el índice de propagación durante el periodo 2016–2023, alcanzando su fase más crítica entre 2020 y 2023, años caracterizados por un mayor riesgo ecológico debido al estrés hídrico y la acumulación de energía térmica. Esta progresión sugiere un deterioro de las condiciones ambientales que favorece la severidad de los incendios, seguida de un ligero descenso en el bienio 2024–2025 que podría atribuirse a la variabilidad climática interanual o a una recuperación ambiental parcial. En conclusión, el sistema muestra una evolución desde condiciones de equilibrio hacia escenarios de mayor vulnerabilidad climática, consolidando al periodo reciente como el de mayor complejidad para la gestión del fuego.

## Mapa de hotspots de incendios 

In [71]:
df_final[["latitud", "longitud", "indice_propagacion"]].head()

,latitud,longitud,indice_propagacion
0,19.309111,-99.788834,1.715070
1,19.877222,-99.875557,1.774486
2,19.596666,-99.432281,1.369911
3,19.002361,-99.367722,1.673714
4,19.574194,-99.395164,1.485263


In [72]:
df_map = df_final.dropna(subset=["latitud", "longitud", "indice_propagacion"])

In [80]:
import folium
from folium.plugins import HeatMap

df_map = df_final.dropna(subset=["latitud", "longitud", "indice_propagacion"])

df_map["peso"] = (
    df_map["indice_propagacion"] - df_map["indice_propagacion"].min()
) / (
    df_map["indice_propagacion"].max() - df_map["indice_propagacion"].min()
)

map_center = [19.0433, -98.1989]

m = folium.Map(location=map_center, zoom_start=7)

heat_data = df_map[["latitud", "longitud", "peso"]].values.tolist()

HeatMap(heat_data, radius=10, blur=15).add_to(m)

m.save("heatmap_incendios.html")

In [81]:
df_hot = df_map[df_map["indice_propagacion"] > df_map["indice_propagacion"].quantile(0.90)]

In [83]:
m2 = folium.Map(location=map_center, zoom_start=7)

HeatMap(df_hot[["latitud", "longitud", "indice_propagacion"]].values.tolist(),
        radius=15, blur=20).add_to(m2)

m2.save("heatmap_incendios_hot.html")

El análisis espacial mediante mapas de calor permitió identificar zonas con alta concentración del índice de propagación, lo que sugiere la existencia de hotspots de incendios forestales. Estas áreas representan regiones donde la combinación de condiciones climáticas y vegetación favorece la rápida expansión del fuego.

## Clustering de hotspots con K-Means geográfico

In [84]:
df_cluster = df_final.dropna(subset=["latitud", "longitud"])

In [85]:
X = df_cluster[["latitud", "longitud"]]

In [86]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=5, random_state=42)
df_cluster["cluster"] = kmeans.fit_predict(X)

In [87]:
import folium

map_center = [19.0433, -98.1989]
m = folium.Map(location=map_center, zoom_start=7)

colors = ["red", "blue", "green", "purple", "orange"]

for _, row in df_cluster.iterrows():
    folium.CircleMarker(
        location=[row["latitud"], row["longitud"]],
        radius=3,
        color=colors[row["cluster"] % len(colors)],
        fill=True,
        fill_opacity=0.7
    ).add_to(m)

m.save("clusters_incendios.html")

In [88]:
df_cluster.groupby("cluster")["indice_propagacion"].mean()

cluster
0    1.662810
1    1.807320
2    1.587966
3    1.813608
4    1.695005
Name: indice_propagacion, dtype: float64

In [89]:
df_cluster["cluster"].value_counts()

cluster
1    3595
2    2548
3    2510
4    2009
0    1924
Name: count, dtype: int64

El análisis de clustering espacial mediante K-Means permitió identificar regiones homogéneas de ocurrencia de incendios forestales. Cada cluster representa una zona con patrones similares de localización, lo que sugiere la existencia de regiones de riesgo diferenciadas dentro del territorio analizado.